# 01 · Profile and preprocess every dataset (protocol v1.0)

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

Runs the frozen preprocessing of `docs/protocol/preregistration.md` on the data in `MyDrive/PhD-AI-Cyber/data/`.
Nothing here trains a model, and nothing is subsampled: every dataset is processed in full.

**How to open:** in Colab, *File → Upload notebook* and pick this file (or, after the first run, open it from
`MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/notebooks/`). Sign in as san60lamichhane@gmail.com.
The `GITHUB_TOKEN` secret you already use must be switched on for this notebook (🔑 panel).

**How to run:** top to bottom, one cell at a time, waiting for each to finish.

- **Part A** (20–40 min): schema profile of all 10 datasets, then CICIDS2017, UNSW-NB15 and NSL-KDD. Then stop
  and report in the project thread.
- **Part B** (several hours, can be split over sessions): CICIoMT2024, ToN_IoT, CIC-IoT2023, CSE-CIC-IDS2018, LANL.
  Use *Runtime → Change runtime type → High-RAM* for CSE-CIC-IDS2018 and CIC-IoT2023.

**Where everything is saved**

| What | Drive (`MyDrive/PhD-AI-Cyber/`) | GitHub |
|---|---|---|
| reports, data cards, schema profile, run logs, frozen feature lists | `code/phd-ai-threat-detection/results/data_audit/`, `docs/data_cards/`, `config/feature_mapping/` | same paths, pushed by the `push(...)` cells |
| cleaned data (parquet) | `data/processed/<dataset>/v1/` | no (data stay in Drive); SHA-256 in the reports |
| frozen splits | `data/splits/<dataset>/` | no; SHA-256 in the reports |


In [ ]:
# Setup: mounts Drive, updates the code in Drive from GitHub, installs packages, runs the tests.
# Expected end: "23 passed". Re-run this cell first after any disconnect.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_NAME  = 'phd-ai-threat-detection'
REPO_DIR   = f'{DRIVE_ROOT}/code/{REPO_NAME}'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'      # stage files on the fast local disk, not in Drive

from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/{REPO_NAME}.git'
# A copy of this notebook uploaded into the repo folder would block the update: park it in backups/ first.
import shutil, subprocess
nb_rel = 'notebooks/01_profile_and_preprocess.ipynb'
tracked = subprocess.run(['git', '-C', REPO_DIR, 'ls-files', '--error-unmatch', nb_rel], capture_output=True).returncode == 0
if not tracked and pathlib.Path(REPO_DIR, nb_rel).exists():
    pathlib.Path(DRIVE_ROOT, 'backups').mkdir(exist_ok=True)
    shutil.move(str(pathlib.Path(REPO_DIR, nb_rel)), f'{DRIVE_ROOT}/backups/01_profile_and_preprocess.uploaded.ipynb')
!git -C "$REPO_DIR" pull --ff-only "$REMOTE" main
assert pathlib.Path(REPO_DIR, 'src/data/pipeline.py').exists(), 'code update failed: read the git message above'
!git -C "$REPO_DIR" config user.name "Sandesh Lamichhane"
!git -C "$REPO_DIR" config user.email "san30lamichhane@gmail.com"
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests

In [ ]:
# Defines push(): commits reports, data cards, logs and feature lists from the Drive copy of the repo
# to GitHub, so Drive and GitHub hold the same records. Just run it; nothing visible happens.
import subprocess

def push(message):
    """Commit the audit results, data cards and frozen feature lists, and push them to GitHub."""
    def git(*args):
        r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
        print((r.stdout + r.stderr).replace(TOKEN, '***').strip())
        return r.returncode
    git('add', 'results/data_audit', 'docs/data_cards', 'config/feature_mapping')
    if git('commit', '-m', message) != 0:
        print('nothing new to commit')
    git('pull', '--rebase', REMOTE, 'main')
    git('push', REMOTE, 'HEAD:main')


## Part A

### A1. Schema profile (samples only, a few minutes)
Checks every dataset's files, columns, labels and timestamps against `config/preprocessing.yaml`.
The table at the end should say `none` under problems for every dataset.

In [ ]:
!python scripts/preprocess.py --profile
from IPython.display import Markdown
Markdown(open('results/data_audit/schema_profile.md').read())

### A2. The three small datasets
CICIDS2017 (≈2.8 M flows), UNSW-NB15 (≈2.5 M), NSL-KDD (≈150 k). About 10–20 minutes in total.

In [ ]:
!python scripts/preprocess.py --dataset cicids2017_improved unsw_nb15 nsl_kdd
Markdown(open('docs/data_cards/cicids2017_improved.md').read())

In [ ]:
push("Schema profile and preprocessing of CICIDS2017, UNSW-NB15, NSL-KDD (protocol v1.0)")

**Stop here** and say in the project thread that Part A is pushed. The profile and the three data cards get
checked against the protocol before the long runs.

## Part B

Each cell is independent. If the session drops, re-run the setup cell and continue with the next dataset.

In [ ]:
!python scripts/preprocess.py --dataset ciciomt2024 ton_iot
push("Preprocess CICIoMT2024 and ToN_IoT (protocol v1.0)")

In [ ]:
# High-RAM runtime recommended. CIC-IoT2023 has ~46 M rows.
!python scripts/preprocess.py --dataset ciciot2023
push("Preprocess CIC-IoT2023 (protocol v1.0)")

In [ ]:
# High-RAM runtime recommended. The corrected 2018 release is the largest dataset (tens of millions of flows).
!python scripts/preprocess.py --dataset csecicids2018_improved
!python scripts/preprocess.py --shared-features
!python scripts/preprocess.py --cross-overlap
push("Preprocess CSE-CIC-IDS2018, freeze the shared CIC feature list, cross-dataset overlap (protocol v1.0)")

In [ ]:
# LANL: streams ~1.6 billion events into host-hour features. 1-3 hours; resumes per source if interrupted.
!python scripts/preprocess.py --dataset lanl_cyber1
push("LANL host-hour features, compromises and Level 3 split (protocol v1.0)")